In [ ]:
#Yesim Nur Tortop
# 911967
# y.tortop@campus.unimib.it


import numpy as np # linear algebra
import pandas as pd # data processing, 
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.api as sm
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.layers import Input, LSTM

****Data Prepocess****

In [ ]:
# Function to load, filter, and save the DataFrame
def process_csv(file_path, output_path, start_date, end_date):
    # Load the CSV file
    df = pd.read_csv(file_path)
    
    # Convert the 'Date' column to datetime format
    df['Date'] = pd.to_datetime(df['Date'], format='%Y/%m/%d %H:%M')
    
    # Filter the DataFrame by date
    filtered_df = df[(df['Date'] >= start_date) & (df['Date'] < end_date)]
    
    # Remove the 'Id' column if it exists
    if 'Id' in filtered_df.columns:
        filtered_df = filtered_df.drop(columns=['Id'])
    
    # Save the filtered DataFrame to a new CSV file
    filtered_df.to_csv(output_path, index=False)
    print(f"Filtered data saved to {output_path}")

# Define the date range
start_date = pd.Timestamp('2016-01-01 00:00')
end_date = pd.Timestamp('2024-04-01 00:00')


file_paths = {
    'global_radiation': '/kaggle/input/data-arpa/DataARPA/global_radiation/global_radiation.csv',
    'rain': '/kaggle/input/data-arpa/DataARPA/rain/rain.csv',
    'humidity': '/kaggle/input/data-arpa/DataARPA/rel_humidity/rel_humidity.csv',
    'temperature': '/kaggle/input/data-arpa/DataARPA/temperature/temperature.csv',
    'wind_direction': '/kaggle/input/data-arpa/DataARPA/wind_direction/wind_direction.csv',
    'wind_speed': '/kaggle/input/data-arpa/DataARPA/wind_speed/wind_speed.csv'
}

# Process each file
for name, file_path in file_paths.items():
    output_path = f'/kaggle/working/{name}_filtered.csv'  # Change output directory to /kaggle/working/
    process_csv(file_path, output_path, start_date, end_date)


In [ ]:
rainnew = pd.read_csv('/kaggle/working/rain_filtered.csv', parse_dates=['Date'])

df_hourly = rainnew.set_index('Date').resample('h').first().reset_index()

output_path = '/kaggle/working/rain_filtered.csv'
df_hourly.to_csv(output_path, index=False)
print(f"Hourly data saved to {output_path}")
rain_new= pd.read_csv('rain_filtered.csv')
rain_new

In [ ]:
# Directory containing the CSV files
directory_path = '/kaggle/working/'

# List of filenames to process
filenames = [
    'wind_speed_filtered.csv',
    'wind_direction_filtered.csv',
    'global_radiation_filtered.csv',
    'rain_filtered.csv',
    'humidity_filtered.csv',
    'temperature_filtered.csv'
]


In [ ]:
# Function to count total rows and occurrences of -999.0 in a DataFrame
def count_rows_and_neg999(file_path):
    df = pd.read_csv(file_path)
    total_rows = len(df)
    neg999_count = (df == -999.0).sum().sum()  # Count -999.0 in all columns
    return total_rows, neg999_count

# Iterate through the CSV files and count rows and -999.0 values
counts = {}
for filename in os.listdir(directory_path):
    if filename.endswith(".csv"):
        file_path = os.path.join(directory_path, filename)
        total_rows, neg999_count = count_rows_and_neg999(file_path)
        counts[filename] = {'total_rows': total_rows, 'neg999_count': neg999_count}

# Print the counts
for filename, count in counts.items():
    print(f"{filename}: Total Rows = {count['total_rows']}, -999.0 Count = {count['neg999_count']}")


In [ ]:
# Function to replace -999.0 with mean of previous and next 5 samples in the second column
def replace_neg999(file_path):
    df = pd.read_csv(file_path)
    if len(df.columns) > 1:
        second_col = df.columns[1]
        
        df[second_col] = pd.to_numeric(df[second_col], errors='coerce')
        
        neg999_indices = df[df[second_col] == -999.0].index
        for idx in neg999_indices:
            start = max(0, idx - 5)
            end = min(len(df), idx + 6)
            valid_values = pd.concat([df[second_col][start:idx], df[second_col][idx + 1:end]])
            valid_values = valid_values[valid_values != -999.0]
            if len(valid_values) > 0:
                mean_value = valid_values.mean()
                df.at[idx, second_col] = mean_value
            else:
                df.at[idx, second_col] = np.nan  
    return df

# Iterate through the CSV files and process them
for filename in filenames:
    file_path = os.path.join(directory_path, filename)
    if os.path.isfile(file_path):
        df = replace_neg999(file_path)
        df.to_csv(file_path, index=False) 
        print(f"Processed {filename}")

# Function to count total rows and occurrences of -999.0 in the second column of a DataFrame
def count_rows_and_neg999(file_path):
    df = pd.read_csv(file_path)
    if len(df.columns) > 1:
        second_col = df.columns[1]
        total_rows = len(df)
        neg999_count = (df[second_col] == -999.0).sum()  # Count -999.0 in the second column
        return total_rows, neg999_count
    else:
        return 0, 0  # If there is no second column, return 0 counts

# Iterate through the specified filenames and count rows and -999.0 values in the second column
counts = {}
for filename in filenames:
    file_path = os.path.join(directory_path, filename)
    if os.path.isfile(file_path):
        total_rows, neg999_count = count_rows_and_neg999(file_path)
        counts[filename] = {'total_rows': total_rows, 'neg999_count': neg999_count}

# Print the counts
for filename, count in counts.items():
    print(f"{filename}: Total Rows = {count['total_rows']}, -999.0 Count = {count['neg999_count']}")


In [ ]:
# Load Wind the data
wind_direction_data = pd.read_csv('/kaggle/working/wind_direction_filtered.csv')
wind_speed_data = pd.read_csv('/kaggle/working/wind_speed_filtered.csv')

# Merge the datasets on the Date column
wind_data = pd.merge(wind_direction_data, wind_speed_data, on='Date', suffixes=('_dir', '_spd'))

# Convert wind direction to radians
wind_data['WindDirection_radians'] = np.deg2rad(wind_data[' wind_d'])

# Calculate sine and cosine components (using radians for correct trigonometry)
wind_data['Wind_X'] = wind_data['WindSpeed'] * np.cos(wind_data['WindDirection_radians'])
wind_data['Wind_Y'] = wind_data['WindSpeed'] * np.sin(wind_data['WindDirection_radians'])

# Handle cases where wind speed is zero
wind_data.loc[wind_data['WindSpeed'] == 0, ['Wind_X', 'Wind_Y']] = 0

# Drop unnecessary columns
wind_data = wind_data.drop(columns=[' wind_d', 'WindSpeed', 'WindDirection_radians'])

# Save the processed data to a new CSV file
wind_data.to_csv('/kaggle/working/combined_wind_data.csv', index=False)

# Plot the transformed wind components for visualization
plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
plt.hist2d(wind_data['Wind_X'], wind_data['Wind_Y'], bins=50, cmap='viridis')
plt.colorbar()
plt.xlabel('Wind X [m/s]')
plt.ylabel('Wind Y [m/s]')
plt.title('Transformed Wind Components')

plt.subplot(1, 2, 2)
plt.hist2d(wind_direction_data[' wind_d'], wind_speed_data['WindSpeed'], bins=50, cmap='viridis')
plt.colorbar()
plt.xlabel('Wind Direction [deg]')
plt.ylabel('Wind Speed [m/s]')
plt.title('Original Wind Data')

plt.tight_layout()
plt.show()


In [ ]:
# New Paths 
filenames = [
    'combined_wind_data.csv',
    'global_radiation_filtered.csv',
    'rain_filtered.csv',
    'humidity_filtered.csv',
    'temperature_filtered.csv'
]

# Normalization
def normalize_columns(file_path):
    df = pd.read_csv(file_path)
    columns_to_normalize = [col for col in df.columns if col != 'Date']
    
    if columns_to_normalize:
        scaler = MinMaxScaler()
        df[columns_to_normalize] = scaler.fit_transform(df[columns_to_normalize])
    
    return df

for filename in filenames:
    file_path = os.path.join(directory_path, filename)
    if os.path.isfile(file_path):
        df = normalize_columns(file_path)
        df.to_csv(file_path, index=False)  # Save the normalized DataFrame back to the same file
        print(f"Normalized {filename}")




In [ ]:
# Function to check if all rows in the first column are the same across all CSVs
def check_first_column_consistency(filenames, directory_path):
    first_column_values = None
    consistent = True

    for filename in filenames:
        file_path = os.path.join(directory_path, filename)
        if os.path.isfile(file_path):
            df = pd.read_csv(file_path, usecols=[0])
            if first_column_values is None:
                first_column_values = df.iloc[:, 0].tolist()
            else:
                if not df.iloc[:, 0].tolist() == first_column_values:
                    print(f"Inconsistency found in {filename}")
                    consistent = False

    if consistent:
        print("All first columns are consistent across all CSV files.")
    else:
        print("There are inconsistencies in the first columns across the CSV files.")

# Run the consistency check
check_first_column_consistency(filenames, directory_path)


In [ ]:

def calculate_monthly_averages(file_path):
    df = pd.read_csv(file_path, parse_dates=[0])
    df.set_index(df.columns[0], inplace=True)
    monthly_averages = df.resample('ME').mean()  # Calculate monthly averages
    return monthly_averages

# Prepare the plot
plt.figure(figsize=(10, 6))

# Iterate through the CSV files and calculate monthly averages
for filename in filenames:
    file_path = os.path.join(directory_path, filename)
    if os.path.isfile(file_path):
        monthly_avg = calculate_monthly_averages(file_path)
        if monthly_avg is not None:
            for column in monthly_avg.columns:
                plt.plot(monthly_avg.index, monthly_avg[column], label=f"{filename} - {column}")

# Customize the plot
plt.title('Monthly Average Values of Columns')
plt.xlabel('Month')
plt.ylabel('Average Value')
plt.legend()
plt.grid(True)

# Show the plot
plt.show()


In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt

# Paths
directory_path = '/kaggle/working/'
file_info = {
    'combined_wind_data.csv': ['Wind_X', 'Wind_Y'],
    'global_radiation_filtered.csv': 'Rad',
    'rain_filtered.csv': 'Prec',
    'humidity_filtered.csv': 'Hum',
    'temperature_filtered.csv': 'Temp'
}

# Function to extract the date and specified columns from each file
def extract_columns(file_path, cols):
    df = pd.read_csv(file_path, parse_dates=[0])
    if len(df.columns) > 1:
        if isinstance(cols, list):
            return df.iloc[:, 0], df[cols]
        else:
            return df.iloc[:, 0], df[[cols]]
    else:
        print(f"{file_path} does not have the specified columns.")
        return None, None

# Create a DataFrame to hold the combined data
combined_df = pd.DataFrame()

# Extract data from each file and append to combined_df
date_series = None
for filename, cols in file_info.items():
    file_path = os.path.join(directory_path, filename)
    if os.path.isfile(file_path):
        date_series, data_series = extract_columns(file_path, cols)
        if date_series is not None and data_series is not None:
            combined_df = pd.concat([combined_df, data_series], axis=1)

# Add the date series as the first column
if date_series is not None:
    combined_df.insert(0, 'Date', date_series)


#Split the data based on the date
split_date = pd.Timestamp('2024-01-01 00:00')
combined_data_test = combined_df[combined_df['Date'] >= split_date] # I use this to make another manueal test 
combined_df = combined_df[combined_df['Date'] < split_date]


# Save to separate CSV files
output_path_train = os.path.join(directory_path, 'combined_data.csv')
output_path_test = os.path.join(directory_path, 'combined_data_test.csv')

combined_df.to_csv(output_path_train, index=False)
combined_data_test.to_csv(output_path_test, index=False)

# Function to calculate monthly average values for all columns except 'Date'
def calculate_monthly_averages(df):
    df.set_index('Date', inplace=True)
    monthly_averages = df.resample('ME').mean()  # Calculate monthly averages
    return monthly_averages

# Calculate monthly averages
monthly_averages = calculate_monthly_averages(combined_data_test)

combined_data_test

In [ ]:

# Load the dataset
df = pd.read_csv('/kaggle/working/combined_data.csv', index_col='Date', parse_dates=True)

# Define the split ratios
train_size = 0.7
val_size = 0.15
test_size = 0.15


assert train_size + val_size + test_size == 1, "The split ratios must sum to 1"


n = len(df)
train_end = int(train_size * n)
val_end = int((train_size + val_size) * n)

# Split the data
train_data = df.iloc[:train_end]
val_data = df.iloc[train_end:val_end]
test_data = df.iloc[val_end:]

# Display the sizes of the splits
print(f"Training set size: {len(train_data)}")
print(f"Validation set size: {len(val_data)}")
print(f"Test set size: {len(test_data)}")

# Function to create sequences
def create_sequences(data, seq_length):
    X = []
    y = []
    for i in range(len(data) - seq_length):
        X.append(data.iloc[i:i+seq_length].values)
        y.append(data.iloc[i+seq_length].values)
    return np.array(X), np.array(y)

# Define sequence length
seq_length = 24  # Using 24 hours (1 day) to predict the next hour

# Create sequences for each split
X_train, y_train = create_sequences(train_data, seq_length)
X_val, y_val = create_sequences(val_data, seq_length)
X_test, y_test = create_sequences(test_data, seq_length)

# Check the shapes of the sequences
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

print(train_data.head())
print(val_data.head())
print(test_data.head())

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
import matplotlib.pyplot as plt


# Define the RNN model
def create_rnn_model(input_shape):
    model = Sequential()
    model.add(SimpleRNN(50, return_sequences=True, input_shape=input_shape))
    model.add(Dropout(0.2))
    model.add(SimpleRNN(50, return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(input_shape[1]))  
    return model

# Define the input shape based on the training data
input_shape = (X_train.shape[1], X_train.shape[2])

# RNN model
modelSRNN = create_rnn_model(input_shape)

modelSRNN.compile(optimizer='adam', loss='mean_squared_error')

modelSRNN.summary()

early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True,min_delta=1e-3)


history = modelSRNN.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_val, y_val), callbacks=[early_stopping])


plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()



In [ ]:
# Define start and end dates for the desired range
start_date = '2023-01-01'
end_date = '2023-01-30'

# Filter the test_data DataFrame to include only the specified date range
filtered_test_data = test_data.loc[start_date:end_date]

# Create sequences for the filtered test data
X_test_data, y_test_data = create_sequences(filtered_test_data, seq_length)

# Make predictions with the model
y_pred_data = modelSRNN.predict(X_test_data)

# Plot actual vs predicted values for each feature within the specified date range
for i, feature in enumerate(filtered_test_data.columns):
    plt.figure(figsize=(18, 6))
    date_range = filtered_test_data.index[seq_length:]
    plt.plot(date_range, y_test_data[:, i], label=f'Actual {feature}')
    plt.plot(date_range, y_pred_data[:, i], label=f'Predicted {feature}')
    plt.title(f'Actual vs Predicted {feature} ({start_date} to {end_date})')
    plt.xlabel('Time')
    plt.ylabel(feature)
    plt.legend()
    plt.show()


In [ ]:
# Seperate Test Data
sep_test_data = pd.read_csv('/kaggle/input/newtestdata/combined_data_test (1).csv', index_col='Date', parse_dates=True)

import pandas as pd
import matplotlib.pyplot as plt

# Define start and end dates for the desired range
start_date = '2024-03-01'
end_date = '2024-03-30'

# Filter the test_data DataFrame to include only the specified date range
filtered_test_data = sep_test_data.loc[start_date:end_date]

# Create sequences for the filtered test data
X_test_data, y_test_data = create_sequences(filtered_test_data, seq_length)

# Make predictions with the model
y_pred_data = modelSRNN.predict(X_test_data)

# Plot actual vs predicted values for each feature within the specified date range
for i, feature in enumerate(filtered_test_data.columns):
    plt.figure(figsize=(18, 6))
    date_range = filtered_test_data.index[seq_length:]
    plt.plot(date_range, y_test_data[:, i], label=f'Actual {feature}')
    plt.plot(date_range, y_pred_data[:, i], label=f'Predicted {feature}')
    plt.title(f'Actual vs Predicted {feature} ({start_date} to {end_date})')
    plt.xlabel('Time')
    plt.ylabel(feature)
    plt.legend()
    plt.show()

 

In [ ]:

# Function to calculate metrics for each feature
def calculate_metrics(y_test, y_pred, feature_names):
    metrics = {}
    for i, feature in enumerate(feature_names):
        mae = mean_absolute_error(y_test[:, i], y_pred[:, i])
        rmse = np.sqrt(mean_squared_error(y_test[:, i], y_pred[:, i]))
        r2 = r2_score(y_test[:, i], y_pred[:, i])
        # Small epsilon avoids division-by-zero when true values are near zero
        eps = 1e-8
        mape = np.mean(np.abs((y_test[:, i] - y_pred[:, i]) / (np.abs(y_test[:, i]) + eps))) * 100
        smape = 100 / len(y_test[:, i]) * np.sum(
            2 * np.abs(y_pred[:, i] - y_test[:, i]) / (np.abs(y_test[:, i]) + np.abs(y_pred[:, i]) + eps)
        )

        metrics[feature] = {
            'MAE': mae,
            'RMSE': rmse,
            'R2': r2,
            'MAPE': mape,
            'sMAPE': smape
        }
        print(f"{feature} - MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}, MAPE: {mape:.2f}%, sMAPE: {smape:.2f}%")

    return metrics


feature_names = test_data.columns.tolist()

# Calculate metrics for each feature
metrics = calculate_metrics(y_test, y_pred, feature_names)

# Plot learning curves
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Learning Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()


In [ ]:

# Define the GRU model
def create_gru_model(input_shape):
    model = Sequential()
    model.add(GRU(50, return_sequences=True, input_shape=input_shape))
    model.add(Dropout(0.2))
    model.add(GRU(50, return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(input_shape[1]))  # Output layer should match the number of features
    return model

# Define the input shape based on the training data
input_shape = (X_train.shape[1], X_train.shape[2])

# Create the GRU model
gru_model = create_gru_model(input_shape)

# Compile the model
gru_model.compile(optimizer='adam', loss='mean_squared_error')

# Print the model summary
gru_model.summary()


In [ ]:
# Define early stopping
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, min_delta=1e-3)

# Train the model
history = gru_model.fit(X_train, y_train, epochs=20, batch_size=128, validation_data=(X_val, y_val), callbacks=[early_stopping])



plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()


In [ ]:
# Create sequences for the test data
X_test, y_test = create_sequences(test_data, seq_length)

# Evaluate the model on the test set
test_loss = gru_model.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss}")

# Make predictions on the test set
y_pred = gru_model.predict(X_test)

In [ ]:
# Define start and end dates for the desired range
start_date = '2023-01-01'
end_date = '2023-01-31'

# Filter the test_data DataFrame to include only the specified date range
filtered_test_data = test_data.loc[start_date:end_date]

# Create sequences for the filtered test data
X_test_data, y_test_data = create_sequences(filtered_test_data, seq_length)

# Make predictions with the model
y_pred_data = gru_model.predict(X_test_data)

# Plot actual vs predicted values for each feature within the specified date range
for i, feature in enumerate(filtered_test_data.columns):
    plt.figure(figsize=(18, 6))
    date_range = filtered_test_data.index[seq_length:]
    plt.plot(date_range, y_test_data[:, i], label=f'Actual {feature}')
    plt.plot(date_range, y_pred_data[:, i], label=f'Predicted {feature}')
    plt.title(f'Actual vs Predicted {feature} ({start_date} to {end_date})')
    plt.xlabel('Time')
    plt.ylabel(feature)
    plt.legend()
    plt.show()


In [ ]:
# Calculate metrics for each feature
metrics = calculate_metrics(y_test, y_pred, feature_names)

# Plot learning curves
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Learning Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()


In [ ]:
# Define the LSTM model
model_LSTM = Sequential()
model_LSTM.add(Input(shape=(seq_length, X_train.shape[2])))  # Define the input shape using Input layer
model_LSTM.add(LSTM(50, return_sequences=True))
model_LSTM.add(Dropout(0.2))
model_LSTM.add(LSTM(50, return_sequences=False))
model_LSTM.add(Dropout(0.2))
model_LSTM.add(Dense(X_train.shape[2]))  # Output layer should match the number of features

# Compile the model
model_LSTM.compile(optimizer='adam', loss='mean_squared_error')

# Print the model summary
model_LSTM.summary()



In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True,min_delta=1e-3)


history = model_LSTM.fit(X_train, y_train, epochs=20, batch_size=128, validation_data=(X_val, y_val), callbacks=[early_stopping])


plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Evaluate the model on the test set
test_loss = model_LSTM.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss}")

# Make predictions
y_pred = model_LSTM.predict(X_test)




In [ ]:
# Define start and end dates for the desired range
start_date = '2023-01-01'
end_date = '2023-01-31'

# Filter the test_data DataFrame to include only the specified date range
filtered_test_data = test_data.loc[start_date:end_date]

# Create sequences for the filtered test data
X_test_data, y_test_data = create_sequences(filtered_test_data, seq_length)

# Make predictions with the model
y_pred_data = model_LSTM.predict(X_test_data)

# Plot actual vs predicted values for each feature within the specified date range
for i, feature in enumerate(filtered_test_data.columns):
    plt.figure(figsize=(18, 6))
    date_range = filtered_test_data.index[seq_length:]
    plt.plot(date_range, y_test_data[:, i], label=f'Actual {feature}')
    plt.plot(date_range, y_pred_data[:, i], label=f'Predicted {feature}')
    plt.title(f'Actual vs Predicted {feature} ({start_date} to {end_date})')
    plt.xlabel('Time')
    plt.ylabel(feature)
    plt.legend()
    plt.show()


In [ ]:

# Assuming you have the feature names in your data
feature_names = test_data.columns.tolist()

# Calculate metrics for each feature
metrics = calculate_metrics(y_test, y_pred, feature_names)

# Plot learning curves
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Learning Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()


In [ ]:
import kerastuner as kt

#  KerasTuner
def build_model(hp):
    model = Sequential()
    model.add(Input(shape=(seq_length, X_train.shape[2])))
    model.add(LSTM(hp.Int('units', min_value=32, max_value=256, step=32), return_sequences=True))
    model.add(Dropout(hp.Float('dropout_rate', min_value=0.2, max_value=0.5, step=0.1)))
    model.add(LSTM(hp.Int('units', min_value=32, max_value=256, step=32), return_sequences=False))
    model.add(Dropout(hp.Float('dropout_rate', min_value=0.2, max_value=0.5, step=0.1)))
    model.add(Dense(X_train.shape[2]))  # Output layer should match the number of features

    model.compile(optimizer=tf.keras.optimizers.Adam(hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')),
                  loss='mean_squared_error')

    return model

# Hyperparameter tuning
tuner = kt.RandomSearch(build_model,
                        objective='val_loss',
                        max_trials=10,
                        executions_per_trial=1,
                        directory='hyperparam_tuning',
                        project_name='lstm_tuning')

# Run the tuner
tuner.search(X_train, y_train, epochs=20, validation_data=(X_val, y_val))

# Get the best model
best_model = tuner.get_best_models(num_models=1)[0]

# Evaluate the model on the test set
test_loss = best_model.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss}")

# Make predictions
y_pred = best_model.predict(X_test)






In [ ]:
# Define the LSTM model with best hyperparameters
model = Sequential()
model.add(Input(shape=(seq_length, X_train.shape[2])))  # Define the input shape using Input layer
model.add(LSTM(50, return_sequences=True, kernel_regularizer=tf.keras.regularizers.l2(0.001)))
model.add(Dropout(0.2))
model.add(LSTM(50, return_sequences=False, kernel_regularizer=tf.keras.regularizers.l2(0.001)))
model.add(Dropout(0.2))
model.add(Dense(X_train.shape[2]))  # Output layer should match the number of features

# Compile the model
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='mean_squared_error')

# Set early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True,min_delta=1e-3)

# Train the model
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_val, y_val), callbacks=[early_stopping])


In [ ]:
# Make predictions with the model
y_pred_data = model.predict(X_test_data)

# Plot actual vs predicted values for each feature within the specified date range
for i, feature in enumerate(filtered_test_data.columns):
    plt.figure(figsize=(18, 6))
    date_range = filtered_test_data.index[seq_length:]
    plt.plot(date_range, y_test_data[:, i], label=f'Actual {feature}')
    plt.plot(date_range, y_pred_data[:, i], label=f'Predicted {feature}')
    plt.title(f'Actual vs Predicted {feature} ({start_date} to {end_date})')
    plt.xlabel('Time')
    plt.ylabel(feature)
    plt.legend()
    plt.show()
metrics = calculate_metrics(y_test, y_pred, feature_names)


In [ ]:

from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# complex LSTM model with best hyperparameters
def complex_lstm_model(input_shape):
    model = Sequential()
    model.add(Input(shape=input_shape))
    model.add(Bidirectional(LSTM(64, return_sequences=True, kernel_regularizer=tf.keras.regularizers.l2(0.001))))
    model.add(Dropout(0.3))
    model.add(Bidirectional(LSTM(64, return_sequences=True, kernel_regularizer=tf.keras.regularizers.l2(0.001))))
    model.add(Dropout(0.3))
    model.add(LSTM(32, return_sequences=False, kernel_regularizer=tf.keras.regularizers.l2(0.001)))
    model.add(Dropout(0.3))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(input_shape[1]))
    return model

# Define the input shape based on the training data
input_shape = (X_train.shape[1], X_train.shape[2])

# Create the complex LSTM model
model = complex_lstm_model(input_shape)

# Compile the model
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='mean_squared_error')

# Print the model summary
model.summary()

# Set early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, min_delta=1e-3)

# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=64, validation_data=(X_val, y_val), callbacks=[early_stopping])



In [ ]:
y_pred_data = model.predict(X_test_data)

# Plot actual vs predicted values for each feature within the specified date range
for i, feature in enumerate(filtered_test_data.columns):
    plt.figure(figsize=(18, 6))
    date_range = filtered_test_data.index[seq_length:]
    plt.plot(date_range, y_test_data[:, i], label=f'Actual {feature}')
    plt.plot(date_range, y_pred_data[:, i], label=f'Predicted {feature}')
    plt.title(f'Actual vs Predicted {feature} ({start_date} to {end_date})')
    plt.xlabel('Time')
    plt.ylabel(feature)
    plt.legend()
    plt.show()
metrics = calculate_metrics(y_test, y_pred, feature_names)